# Reward Model

In [ ]:
import os
import sys
from pathlib import Path

CWD = os.path.realpath(os.getcwd())
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

DATA_DIR = Path(PARENT_DIR).parent / 'data'

In [ ]:
import torch
torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.data import PreferenceDataset, RewardDataCollator

train_data = PreferenceDataset(split="train")
val_data = PreferenceDataset(split="validation")

In [ ]:
from src.modules.reward import RewardModel

MODEL_NAME = "Qwen/Qwen3.5-2B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = RewardModel(MODEL_NAME)
model = model.to(device)

In [ ]:
train_loader = DataLoader(
    train_data,
    batch_size=4,
    shuffle=True,
    collate_fn=RewardDataCollator(
        tokenizer=tokenizer,
        max_length=1024,
    ),
)
val_loader = DataLoader(
    val_data,
    batch_size=4,
    shuffle=False,
    collate_fn=RewardDataCollator(
        tokenizer,
        max_length=1024,
    ),
)

In [ ]:
from torch import optim
from torch.nn import functional as F

EPOCH_SIZE = 200
LEARNING_RATE = 1e-4
EVAL_INTERVAL = 10
params    = [p for n, p in model.named_parameters() if p.requires_grad ]
optimizer = optim.AdamW(params=params, weight_decay=1e-2, lr=LEARNING_RATE)

In [ ]:
def evaluate(model, data_loader, device, max_batches=10):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.inference_mode():
        for batch_index, batch in enumerate(data_loader):
            if batch_index >= max_batches:
                break

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            chosen_rewards = model(
                input_ids=batch["chosen_input_ids"],
                attention_mask=batch["chosen_attention_mask"],
            )

            rejected_rewards = model(
                input_ids=batch["rejected_input_ids"],
                attention_mask=batch["rejected_attention_mask"],
            )

            differences = chosen_rewards - rejected_rewards

            total_loss += (
                -F.logsigmoid(differences).sum().item()
            )
            total_correct += (differences > 0).sum().item()
            total_examples += differences.numel()

    return (
        total_loss / total_examples,
        total_correct / total_examples,
    )

In [ ]:
step = 0
for batch in train_loader:
    step += 1
    if step > EPOCH_SIZE:
        break

    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    model.train()

    ## remember!!! in paper its loss(rθ) = −E(x,y0,y1,i)∼D[log(σ(rθ(x, yi) − rθ(x, y1−i)))]
    ## so minus log then sigmoid the difference between the chosen reward valu and the rejected reward value
    chosen_rewards = model(
        input_ids=batch["chosen_input_ids"],
        attention_mask=batch["chosen_attention_mask"],
    )
    rejected_rewards = model(
        input_ids=batch["rejected_input_ids"],
        attention_mask=batch["rejected_attention_mask"],
    )

    train_loss = -F.logsigmoid(
        chosen_rewards - rejected_rewards
    ).mean()

    optimizer.zero_grad(set_to_none=True)
    train_loss.backward()
    optimizer.step()

    if step % EVAL_INTERVAL == 0 or step == EPOCH_SIZE:
        val_loss, val_accuracy = evaluate(
            model,
            val_loader,
            device,
            max_batches=10,
        )

        print(
            f"step {step}: "
            f"train loss={train_loss.item():.4f}, "
            f"val loss={val_loss:.4f}, "
            f"val accuracy={val_accuracy:.2%}"
        )

In [ ]:
state_dict = model.state_dict()
torch.save(state_dict, '../model/reward_no_lora.pt')
